In [2]:
#LA API QUE SE REGISTRO SON LOS DATOS BASICOS DE LOS JUGADORES PARTICIPANTES
import requests
import pandas as pd

#SE HACE EL LLAMADO DE LA API DESDE UN APARTADO "SEGURO" COMO LO ES ESPN
base_url = "https://site.api.espn.com"
endpoint = "/apis/site/v2/sports/golf/pga/scoreboard?dates=20260816"
response = requests.get(base_url + endpoint)
print(response.status_code) # rectificamos que la aprobacion de la api fue la correcta 
data = response.json() # Convertimos la lectura a JSON para poder emepzar a interpretar 

print(data.keys()) # Miramos todo lo relacionado a lo que contiene el dataset

# ====================== EVENTO ======================================
#NOS ASEGURAMOS DEL TORNEO TRATADO - ESTE TORNEO TIENE BUENAS METRICAS Y OCURRIO HACE POCO
assert data["events"], "No hay eventos disponibles en la API en este momento" # Nos aseguramos de que haya al menos un torneo antes de indexar
evento = data["events"][0] # Ingreso al primer evento que se registro
print(evento)
print(evento["name"])# Tengo el nombre del evento
print(evento["season"])

evento_id = evento["id"]
print("ID del torneo:", evento_id)

competicion = evento["competitions"][0]
print(competicion.keys()) #Miramos las llaves que posee para asi hacer los llamados correspondientes 

jugador = competicion["competitors"]
print(len(jugador)) # Cantidad de jugadores del torneo

print(jugador[-1]["athlete"]["displayName"]) # Probamos que los jugadores esten dentro de la lista (usamos -1 para no depender de un numero fijo de jugadores)
print(jugador[-1]["score"])
print(jugador[0].keys())
print(jugador[0]["linescores"])#comprobamos los scores de un jugador preciso en la lista para corroborar su veracidad

# ================== CREACION DATAFRAME ============================
datos = []

for jugadores in jugador:

    nombre = jugadores["athlete"]["displayName"]

    # Score acumulado del jugador
    score = jugadores.get("score")

    # Posición
    posicion = jugadores.get("order")

    # Resultados por ronda
    linescores = jugadores.get("linescores", [])

    #Temporada
    season = evento["season"]["year"]

    ronda1 = linescores[0].get("value") if len(linescores) > 0 else None
    ronda2 = linescores[1].get("value") if len(linescores) > 1 else None
    ronda3 = linescores[2].get("value") if len(linescores) > 2 else None
    ronda4 = linescores[3].get("value") if len(linescores) > 3 else None


    datos.append(
        {
            "jugador": nombre,
            "posicion": posicion,
            "score": score,
            "ronda1": ronda1,
            "ronda2": ronda2,
            "ronda3": ronda3,
            "ronda4": ronda4,
            "Temporada" :season,
        }
    )

df = pd.DataFrame(datos)
df


df.to_csv("FedEx St. Jude Championship_General.csv", index=False)

200
dict_keys(['leagues', 'season', 'day', 'events', 'provider'])
{'id': '401811962', 'uid': 's:1100~l:1106~e:401811962', 'date': '2026-08-13T04:00Z', 'endDate': '2026-08-16T04:00Z', 'name': 'FedEx St. Jude Championship', 'shortName': 'FedEx St. Jude Championship', 'season': {'year': 2026, 'type': 2, 'slug': 'regular-season'}, 'competitions': [{'id': '401811962', 'uid': 's:1100~l:1106~e:401811962~c:401811962', 'date': '2026-08-13T04:00Z', 'endDate': '2026-08-16T04:00Z', 'timeValid': False, 'neutralSite': False, 'conferenceCompetition': False, 'playByPlayAvailable': False, 'recent': False, 'competitors': [{'id': '9478', 'uid': 's:1100~l:1106~a:9478', 'type': 'athlete', 'order': 1, 'athlete': {'fullName': 'Scottie Scheffler', 'displayName': 'Scottie Scheffler', 'shortName': 'S. Scheffler', 'flag': {'href': 'https://a.espncdn.com/i/teamlogos/countries/500/usa.png', 'alt': 'USA', 'rel': ['country-flag']}}, 'score': '-17', 'linescores': [{'value': 68.0, 'displayValue': '-2', 'period': 1, 'l

In [6]:
#INCORPORO ESTA API PARA PODER ASOCIAR LOS GOLPES EN CADA HOYO Y ASI PODER INCORPORAR UNA METRICA MAS A FONDO
import requests
import pandas as pd

# URL de la API
base_url = "https://site.web.api.espn.com"
endpoint = "/apis/site/v2/sports/golf/pga/leaderboard/401811962/playersummary"

#NECESITO OBTENER LOS IDS REGISTRADOS PARA CADA JUGADOR Y ASI PODER HACER EL LLAMADO DENTRO DE LA API
jugadores = [
    9478,      # 1  Scottie Scheffler
    7081,      # 2  Si Woo Kim
    3832,      # 3  Alex Noren
    9938,      # 4  Sam Burns
    11119,     # 5  Wyndham Clark
    11382,     # 6  Sungjae Im
    4375972,   # 7  Ludvig Åberg
    10140,     # 8  Xander Schauffele
    5860,      # 9  Hideki Matsuyama
    5539,      # 10 Tommy Fleetwood
    9843,      # 11 Jake Knapp
    5338,      # 12 Bud Cauley
    9037,      # 13 Matt Fitzpatrick
    6007,      # 14 Patrick Cantlay
    4408316,   # 15 Nico Echavarria
    388,       # 16 Adam Scott
    10364,     # 17 Kurt Kitayama
    4837368,   # 18 Pierceson Coody
    6825,      # 19 Patrick Rodgers
    9530,      # 20 Maverick McNealy
    5467,      # 21 Jordan Spieth
    4364873,   # 22 Viktor Hovland
    9484,      # 23 Alex Smalley
    8973,      # 24 Max Homa
    4589438,   # 25 Harry Hall
    5076021,   # 26 Ryan Gerard
    5215013,   # 27 Jackson Koivun
    10505,     # 28 J.T. Poston
    5409,      # 29 Russell Henley
    5054388    # 30 Jacob Bridgeman
]

#TENGO UNA LISTA VACIA QUE ME AYUDARA A ALMACENAR LOS DATOS DEL DATAFRAME
todos_los_jugadores = []

#GENERO UN CICLO FOR PARA PODER INCORPORAR TODOS LOS JUGADORES PARA NO HACERLO UNO POR UNO  
for player_id in jugadores:
    # Consulta (una por jugador)
    response = requests.get(
        base_url + endpoint,
        params={
            "season": 2026,
            "player": player_id
        }
    )
    print("Leido con exito", player_id,"-", response.status_code)

    data = response.json()

    # Información del jugador
    profile = data["profile"]

    player = {
        "Jugador_Pro": profile["displayName"],
        "Edad": profile["age"],
        "Ganancia_oficial_2026":profile["earnings"],
        "Ranking" : profile["rank"],
    }

    # Rondas y hoyos
    for ronda in data["rounds"]:
        numero_ronda = ronda["period"]

        for hoyo in ronda["linescores"]:
            numero_hoyo = int(hoyo["period"])
            golpes = int(hoyo["value"])
            display = hoyo["par"]
            num = hoyo["scoreType"]
            player[f"R{numero_ronda}_H{numero_hoyo}_par({display})"] = golpes

    todos_los_jugadores.append(player)

# DataFrame con todos los jugadores juntos
df = pd.DataFrame(todos_los_jugadores)
df


#df.to_csv("FedEx St. Jude Championship_Jugador.csv", index=False)

Leido con exito 9478 - 200
Leido con exito 7081 - 200
Leido con exito 3832 - 200
Leido con exito 9938 - 200
Leido con exito 11119 - 200
Leido con exito 11382 - 200
Leido con exito 4375972 - 200
Leido con exito 10140 - 200
Leido con exito 5860 - 200
Leido con exito 5539 - 200
Leido con exito 9843 - 200
Leido con exito 5338 - 200
Leido con exito 9037 - 200
Leido con exito 6007 - 200
Leido con exito 4408316 - 200
Leido con exito 388 - 200
Leido con exito 10364 - 200
Leido con exito 4837368 - 200
Leido con exito 6825 - 200
Leido con exito 9530 - 200
Leido con exito 5467 - 200
Leido con exito 4364873 - 200
Leido con exito 9484 - 200
Leido con exito 8973 - 200
Leido con exito 4589438 - 200
Leido con exito 5076021 - 200
Leido con exito 5215013 - 200
Leido con exito 10505 - 200
Leido con exito 5409 - 200
Leido con exito 5054388 - 200


,Jugador_Pro,Edad,Ganancia_oficial_2026,Ranking,R1_H1_par(4),R1_H2_par(4),R1_H3_par(5),R1_H4_par(3),R1_H5_par(4),R1_H6_par(4),...,R4_H9_par(4),R4_H10_par(4),R4_H11_par(3),R4_H12_par(4),R4_H13_par(4),R4_H14_par(3),R4_H15_par(4),R4_H16_par(5),R4_H17_par(4),R4_H18_par(4)
0,Scottie Scheffler,30,"$20,937,524",1st,4,4,4,3,4,4,...,4,4,2,4,3,3,4,4,4,4
1,Si Woo Kim,31,"$9,735,444",6th,4,4,4,3,5,4,...,4,4,2,4,4,3,4,5,4,4
2,Alex Noren,44,"$4,275,215",38th,4,4,5,4,4,4,...,5,3,3,4,4,2,3,4,4,5
3,Sam Burns,30,"$8,896,540",8th,4,4,4,3,4,4,...,4,5,3,4,5,2,4,4,4,5
4,Wyndham Clark,32,"$14,236,186",4th,3,4,5,2,5,4,...,4,4,2,4,4,3,4,4,4,3
5,Sungjae Im,28,"$3,730,077",46th,4,4,4,3,3,3,...,4,5,3,4,3,4,4,5,4,5
6,Ludvig Åberg,26,"$7,147,271",13th,4,4,5,3,4,4,...,4,4,3,4,4,3,4,4,4,4
7,Xander Schauffele,32,"$7,452,725",12th,4,5,5,3,4,3,...,4,4,3,4,4,3,3,5,3,4
8,Hideki Matsuyama,34,"$5,679,457",22nd,4,5,5,4,4,4,...,4,4,5,4,3,3,4,5,4,4
9,Tommy Fleetwood,35,"$7,054,605",14th,3,4,3,2,4,4,...,4,4,3,5,5,3,5,4,3,5
